## HW 4: Python for Data Cleaning and Analysis ##
- We will partially replicate HW 2 (Accounting Information and Stock Price) in this exercise using Python.
- Overall, we want to estimate the regression: P = b0 + b1xE + b2xBV + b3xRD + b4xAD + error, where P is stock price, E is earnings, BV is book value, RD is R&D expenses, and AD is advertising expenses.
- Download Compustat data from Canvas, which contains useful data items such as:
- Company ID (gvkey)
- Fiscal year (fyear)
- Total assets (at)
- income before extraordinary items (ib)
- Common equity (ceq)
- Closing stock price (prcc_f)
- Common shares outstanding (csho)
- SIC industry classification code (sic)
- R&D expenses (xrd)
- Advertising expenses (xad)
- Stock exchange code (exchg)
- Run your code before submission to ensure your code works. Always print the variables/lists/DataFrames you create to check if your code works. Your submission should include the cell outputs.
- It is helpful to add, for example, head() or info() to check your data after each step.
- For submission, click "File" and then "Save and Export Notebook As HTML." Submit your HTML file to Canvas.

IMPORTANT: Before we start data analysis, please import the relevant libraries.

In [13]:
# from data cleaning classwork

import pandas as pd
import numpy as np
import datetime as dt

# from data analysis classwork

from scipy import stats
import statsmodels.formula.api as sf


1. Import data to Python. Use read_csv() method. It is helpful to include index_col = False.

In [14]:
# function so i only type the file name (works on both computers) [I made cause I don't keep stuff in same folder]
def load_data(file_name, rows=5):
    # check the end of the file name to pick the right reader
    if file_name.endswith('.csv'):
        reader = pd.read_csv
    else:
        reader = pd.read_excel
    # try personal computer path first, then school desktop
    try:
        df = reader('C:\\Users\\Henry\\OneDrive - University of Virginia\\COMM Semester 3\\1. Accounting Analytics\\Data_Files\\' + file_name, index_col=False)
    except FileNotFoundError:
        df = reader('C:\\Users\\fbw4vt\\OneDrive - University of Virginia\\COMM Semester 3\\1. Accounting Analytics\\Data_Files\\' + file_name, index_col=False)
    # show the first rows (5 unless i give a different number)
    display(df.head(rows))
    return df

data = load_data('HW4.csv')

,costat,curcd,datafmt,indfmt,consol,tic,fyear,gvkey,exchg,sic,at,ceq,ib,xad,xrd,au,csho,prcc_f
0,A,USD,STD,INDL,C,AIR,2001,1004,11.0,5080,710.199,310.235,-58.939,NaN,NaN,6.0,31.870,11.44
1,I,USD,STD,INDL,C,ADCT.1,2001,1013,14.0,3661,2499.700,1893.400,-1287.700,NaN,278.60,1.0,792.000,4.55
2,I,USD,STD,INDL,C,ALO.2,2001,1034,11.0,2834,2390.008,891.616,-35.674,NaN,86.65,7.0,44.318,26.45
3,I,USD,STD,INDL,C,AEN.2,2001,1038,12.0,7830,1279.170,258.181,-11.468,NaN,NaN,7.0,33.819,13.64
4,A,USD,STD,INDL,C,AAL,2001,1045,14.0,4512,32841.000,5373.000,-1762.000,202.0,NaN,4.0,154.485,22.30


2. Drop rows with missing at, prcc_f, csho, ib, and ceq.

In [15]:
data = data.dropna(subset=['prcc_f','csho','ib','ceq']) # from data cleaning lab step 6
data.head()

,costat,curcd,datafmt,indfmt,consol,tic,fyear,gvkey,exchg,sic,at,ceq,ib,xad,xrd,au,csho,prcc_f
0,A,USD,STD,INDL,C,AIR,2001,1004,11.0,5080,710.199,310.235,-58.939,NaN,NaN,6.0,31.870,11.44
1,I,USD,STD,INDL,C,ADCT.1,2001,1013,14.0,3661,2499.700,1893.400,-1287.700,NaN,278.60,1.0,792.000,4.55
2,I,USD,STD,INDL,C,ALO.2,2001,1034,11.0,2834,2390.008,891.616,-35.674,NaN,86.65,7.0,44.318,26.45
3,I,USD,STD,INDL,C,AEN.2,2001,1038,12.0,7830,1279.170,258.181,-11.468,NaN,NaN,7.0,33.819,13.64
4,A,USD,STD,INDL,C,AAL,2001,1045,14.0,4512,32841.000,5373.000,-1762.000,202.0,NaN,4.0,154.485,22.30


3. Drop rows if SIC (assuming it is integer or float) is between 6000 and 6999. That is, we want to exclude financial service firms.

In [16]:
data = data[(data['sic'] < 6000) | (data['sic'] > 6999)] # e.g. keep values below 6000 and above 6999
data.head()

,costat,curcd,datafmt,indfmt,consol,tic,fyear,gvkey,exchg,sic,at,ceq,ib,xad,xrd,au,csho,prcc_f
0,A,USD,STD,INDL,C,AIR,2001,1004,11.0,5080,710.199,310.235,-58.939,NaN,NaN,6.0,31.870,11.44
1,I,USD,STD,INDL,C,ADCT.1,2001,1013,14.0,3661,2499.700,1893.400,-1287.700,NaN,278.60,1.0,792.000,4.55
2,I,USD,STD,INDL,C,ALO.2,2001,1034,11.0,2834,2390.008,891.616,-35.674,NaN,86.65,7.0,44.318,26.45
3,I,USD,STD,INDL,C,AEN.2,2001,1038,12.0,7830,1279.170,258.181,-11.468,NaN,NaN,7.0,33.819,13.64
4,A,USD,STD,INDL,C,AAL,2001,1045,14.0,4512,32841.000,5373.000,-1762.000,202.0,NaN,4.0,154.485,22.30


4. Restrict exchg to values 11, 12, and 14 (i.e., NYSE, NYSE American, and Nasdaq).

In [17]:
exchanges = [11, 12, 14]

data = data[data['exchg'].isin(exchanges)] # using isin() function, not from the lab (but easier instead of having to use a bunch of |)
data.head()

,costat,curcd,datafmt,indfmt,consol,tic,fyear,gvkey,exchg,sic,at,ceq,ib,xad,xrd,au,csho,prcc_f
0,A,USD,STD,INDL,C,AIR,2001,1004,11.0,5080,710.199,310.235,-58.939,NaN,NaN,6.0,31.870,11.44
1,I,USD,STD,INDL,C,ADCT.1,2001,1013,14.0,3661,2499.700,1893.400,-1287.700,NaN,278.60,1.0,792.000,4.55
2,I,USD,STD,INDL,C,ALO.2,2001,1034,11.0,2834,2390.008,891.616,-35.674,NaN,86.65,7.0,44.318,26.45
3,I,USD,STD,INDL,C,AEN.2,2001,1038,12.0,7830,1279.170,258.181,-11.468,NaN,NaN,7.0,33.819,13.64
4,A,USD,STD,INDL,C,AAL,2001,1045,14.0,4512,32841.000,5373.000,-1762.000,202.0,NaN,4.0,154.485,22.30


5. Replace missing xrd and xad with 0. We didn't discuss this in class, please use Google or AI for help.

In [18]:
# replace missing xrd and xad with 0 (fillna is a pandas method, not from the labs)
data['xrd'] = data['xrd'].fillna(0)
data['xad'] = data['xad'].fillna(0)
data.head()

,costat,curcd,datafmt,indfmt,consol,tic,fyear,gvkey,exchg,sic,at,ceq,ib,xad,xrd,au,csho,prcc_f
0,A,USD,STD,INDL,C,AIR,2001,1004,11.0,5080,710.199,310.235,-58.939,0.0,0.00,6.0,31.870,11.44
1,I,USD,STD,INDL,C,ADCT.1,2001,1013,14.0,3661,2499.700,1893.400,-1287.700,0.0,278.60,1.0,792.000,4.55
2,I,USD,STD,INDL,C,ALO.2,2001,1034,11.0,2834,2390.008,891.616,-35.674,0.0,86.65,7.0,44.318,26.45
3,I,USD,STD,INDL,C,AEN.2,2001,1038,12.0,7830,1279.170,258.181,-11.468,0.0,0.00,7.0,33.819,13.64
4,A,USD,STD,INDL,C,AAL,2001,1045,14.0,4512,32841.000,5373.000,-1762.000,202.0,0.00,4.0,154.485,22.30


6. Create the following variables:
- E, defined as ib divided by csho
- BV, defined as ceq divided by csho
- RD, defined as xrd divided by csho
- AD, defined as xad divided by csho

In [19]:
# basic continuous variables (same way as dta = lt / at in the data cleaning lab)
data['E'] = data['ib'] / data['csho']
data['BV'] = data['ceq'] / data['csho']
data['RD'] = data['xrd'] / data['csho']
data['AD'] = data['xad'] / data['csho']
data.head()

,costat,curcd,datafmt,indfmt,consol,tic,fyear,gvkey,exchg,sic,...,ib,xad,xrd,au,csho,prcc_f,E,BV,RD,AD
0,A,USD,STD,INDL,C,AIR,2001,1004,11.0,5080,...,-58.939,0.0,0.00,6.0,31.870,11.44,-1.849357,9.734390,0.000000,0.00000
1,I,USD,STD,INDL,C,ADCT.1,2001,1013,14.0,3661,...,-1287.700,0.0,278.60,1.0,792.000,4.55,-1.625884,2.390657,0.351768,0.00000
2,I,USD,STD,INDL,C,ALO.2,2001,1034,11.0,2834,...,-35.674,0.0,86.65,7.0,44.318,26.45,-0.804955,20.118597,1.955188,0.00000
3,I,USD,STD,INDL,C,AEN.2,2001,1038,12.0,7830,...,-11.468,0.0,0.00,7.0,33.819,13.64,-0.339099,7.634200,0.000000,0.00000
4,A,USD,STD,INDL,C,AAL,2001,1045,14.0,4512,...,-1762.000,202.0,0.00,4.0,154.485,22.30,-11.405638,34.780076,0.000000,1.30757


7. Winsorize the four variables you just created as well as stock price (prcc_f).

In [20]:
# winsorization (same as data cleaning lab, just swapped in variables)
varlist = ['E','BV','RD','AD','prcc_f']
high=.99
low=.01
# replace values above the 99th percentile with the 99th percentile value
for x in varlist:
    data.loc[(data[x]>data[x].quantile(high, interpolation='nearest')),x]=data[x].quantile(high, interpolation='nearest')
# replace values below the 1st percentile with the 1st percentile value
for x in varlist:
    data.loc[(data[x]<data[x].quantile(low, interpolation='nearest')), x]=data[x].quantile(low, interpolation='nearest')

# check that min/max got pulled in
data[varlist].describe([0.01,0.99])

,E,BV,RD,AD,prcc_f
count,68302.000000,68302.000000,68300.000000,68301.000000,68302.000000
mean,1.197794,13.330472,0.585221,0.259344,38.174105
std,3.300580,14.508665,1.108362,0.698248,50.603773
min,-9.007827,-7.747820,0.000000,0.000000,1.400000
1%,-9.007763,-7.747691,0.000000,0.000000,1.400000
50%,0.776347,9.086715,0.000000,0.000000,22.810000
99%,16.913369,82.512714,6.429589,4.379506,331.735000
max,16.913569,82.513031,6.429589,4.379506,331.740000


8. Run the regression: P = b0 + b1xE + b2xBV + b3xRD + b4xAD + error, for the full sample and show the results.

In [21]:
# linear regression, same 3 steps as the data analysis lab (P is the stock price, prcc_f)
model1 = sf.ols(formula = 'prcc_f ~ E + BV + RD + AD', data=data) 
res1 = model1.fit() 
res1.summary() 

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 prcc_f   R-squared:                       0.553
Model:                            OLS   Adj. R-squared:                  0.553
Method:                 Least Squares   F-statistic:                 2.114e+04
Date:                Thu, 22 Oct 2026   Prob (F-statistic):               0.00
Time:                        16:07:14   Log-Likelihood:            -3.3741e+05
No. Observations:               68300   AIC:                         6.748e+05
Df Residuals:                   68295   BIC:                         6.749e+05
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      6.9872      0.187     37.390      0.000       6.621       7.353
E              6.4159      0.047    135.772      0.000       6.323       6.509
BV             1.0884      0.011    100.426      0.000       1.067       1.110
RD            12.7858      0.118    107.972      0.000      12.554      13.018
AD             5.8445      0.188     31.076      0.000       5.476       6.213
==============================================================================
Omnibus:                    38353.776   Durbin-Watson:                   1.872
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           608369.420
Skew:                           2.375   Prob(JB):                         0.00
Kurtosis:                      16.828   Cond. No.                         31.3
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

9. Run the regression individually for each year, and importantly, report R_squared for each year.

In [22]:
# run regressions by year, same groupby/apply way as the data analysis lab (just swapped in my variables)
def groupreg(data, formula):
    model = sf.ols(formula, data=data).fit()
    return pd.Series({
        'E_coef': model.params.get('E'),
        'E_pvalue': model.pvalues.get('E'),
        'BV_coef': model.params.get('BV'),
        'BV_pvalue': model.pvalues.get('BV'),
        'RD_coef': model.params.get('RD'),
        'RD_pvalue': model.pvalues.get('RD'),
        'AD_coef': model.params.get('AD'),
        'AD_pvalue': model.pvalues.get('AD'),
        'n': model.nobs,
        'r2': model.rsquared
    })

group_results = (
    data.groupby('fyear')
         .apply(groupreg, "prcc_f ~ E + BV + RD + AD", include_groups=False)
         .reset_index()
)

print(group_results.round(5))

    fyear   E_coef  E_pvalue  BV_coef  BV_pvalue   RD_coef  RD_pvalue  \
0    2001  2.84145       0.0  1.05880        0.0   5.35023        0.0   
1    2002  3.42716       0.0  1.09433        0.0   3.08394        0.0   
2    2003  3.59742       0.0  1.14286        0.0   4.54876        0.0   
3    2004  3.46812       0.0  1.05559        0.0   3.52245        0.0   
4    2005  4.44898       0.0  0.97784        0.0   3.60763        0.0   
5    2006  4.75282       0.0  0.99232        0.0   4.71082        0.0   
6    2007  6.07214       0.0  0.95108        0.0   4.86011        0.0   
7    2008  2.43358       0.0  1.00429        0.0   3.82074        0.0   
8    2009  4.28675       0.0  1.14777        0.0   4.64540        0.0   
9    2010  5.47354       0.0  0.95960        0.0   3.52266        0.0   
10   2011  6.41807       0.0  0.76224        0.0   2.63724        0.0   
11   2012  5.98739       0.0  0.91686        0.0   5.18628        0.0   
12   2013  6.92708       0.0  0.88777        0.0   

10. In the cell below, please report: which application is easier for this regression analysis: Alteryx or Python?

In [23]:
# It's so much easier in Python! Way faster, much easier to audit and see everything (surprising because it's in code), and way more flexible. 
# It's just a steeper learning curve, tho, where you have to either ask AI or the Prof. to do smt instead of intuitively looking at Alteryx.
# I think had we done Alteryx second and Python first, my answer may be a bit different, too - there's some bias with us knowing the concepts already.